###**Section 2 : Python data handling**

In this section we will:

1. Load the entire 2022 raw energy data from the official RTE website
2. Clean it to make it ready for analysis
3. Create the first basic variables (total production, total consumption and the imbalance between the two) that will serve as foundations for the digital twin built in later sections

In [ ]:
#Link the colab to google drive to access the data
from google.colab import drive
drive.mount('/content/drive')

**1. Data loading and exploration**

We start by loading the raw dataset and exploring it to understand the structure, identify the variables it contains, and determine what cleaning steps are needed before analysis.

In [ ]:
#Import libraries
import pandas as pd
import matplotlib.pyplot as plt

#Load raw RTE dataset (annual electricity data for 2022)
data_2022 = pd.read_csv('/content/drive/MyDrive/Capstone Project - Data & Energy/Project Energy System/Data/eCO2mix_RTE_Annuel-Definitif_2022.csv')

In [ ]:
#Preview dataset information
data_2022.info()

In [ ]:
#Preview dataset statistics
data_2022.describe()

In [ ]:
#Preview dataset structure
data_2022.head()

**2. Understanding the variables**

The meaning of the variables is described in the document "Spécification des fichiers de données en puissance pour éCO2mix" (pages 3–5).

**2.1. Physical meaning:**

- All production and consumption variables are expressed in MW (megawatts), representing electrical power at a given time t. These are average power values measured over a 15-minute interval, which is the temporal resolution of the dataset.

**2.2. State vs Flux:**

- States (instantaneous snapshots at time t): Consommation, Nucléaire, Gaz, Eolien, Solaire, Hydraulique, Ech. physiques, Taux de Co₂.

- Fluxes (variations between two instants): The change in any state variable between t and t+1 (e.g., consumption increasing from 50,000 to 55,000 MW): not present in the raw dataset.

- Instantaneous vs Cumulated:All variables in this dataset are instantaneous power values (MW), not cumulated energy (MWh).

- Forecasts:Prévision J-1 and Prévision J represent day-ahead and same-day consumption forecasts.


**3. Data cleaning**

- The raw dataset contains typical structural issues (unnamed columns, inconsistent naming, missing values, and separated date/time fields).

- Column names were standardized and irrelevant columns were removed or renamed when necessary.

- The `Date` column was converted to datetime format, and basic checks (min/max) were performed to validate the covered period.

- A unified timestamp was created by combining `Date` and `Heures`, then set as the index to transform the dataset into a proper time series. The original columns were removed.


In [ ]:
#Convert Date column to datetime format
data_2022["Date"]=pd.to_datetime(data_2022["Date"])

#Verify the datetime format
print(data_2022.info())

In [ ]:
#Verify dataset covers full year using min/max timestamps
print(data_2022["Date"].min())
print(data_2022["Date"].max())

In [ ]:
#Verify how often each day is repeated in the dataframe
data_2022["Date"].value_counts()

**3 - Analysis #1**

We can observe that each day appears 96 times because measurements are made daily at an interval of 15 min (4 times per hour): 24 hours x 4 measurements/hours = 96 measurements.
The dataset has a 15-minute temporal structure. However, some variables are only recorded every 30 minutes (i.e. Consommation), which results in missing values (NaN) in intermediate timestamps.

In [ ]:
# Standardize column names by removing leading/trailing spaces
data_2022.columns = data_2022.columns.str.strip()

In [ ]:
#Data cleaning

#Rename the columns missing headings in the csv file using the explanation provided in "Spécification des fichiers de données en puissance pour éCO2mix"

data_2022 = data_2022.rename(columns={'Unnamed: 5': 'Prévision J-1', 'Unnamed: 6' : 'Prévision J','Unnamed: 10' :'Nucléaire',
                               'Unnamed: 15' : 'Bioénergies','Unnamed: 24' : 'Fioul - Cogén.', 'Unnamed: 27' : 'Gaz - Cogén.', 'Unnamed: 30' : 'Hydraulique - Fil de l’eau + éclusée',
                               'Unnamed: 33':'Bioénergies- Déchets', 'Unnamed: 34' : 'BioénergiesBiomasse', 'Unnamed: 35' : 'Bioénergies- Biogaz',
                               'Unnamed: 37': 'Déstockage batteries'})
data_2022

In [ ]:
# Remove non-informative columns
data_2022 = data_2022.drop(["Nature", "Unnamed: 0"], axis =1)

In [ ]:
# Remove rows where consumption is missing
data_2022 = data_2022.dropna(subset='Consommation')

**3 - NOTE**

Since the column 'Consommation' is a crucial variable in the analysis, representing the system's demand, any row with missing consumption data cannot be exploited meaningfully.
Therefore, all rows with NaN values in 'Consommation' have been dropped.

In [ ]:
#Create unified timestamp by combining Date and Heures
data_2022["timestamp"] = pd.to_datetime(data_2022["Date"].astype(str) + " " + data_2022["Heures"])


In [ ]:
#Set timestamp as index to convert dataset into a time series
data_2022 = data_2022.set_index(["timestamp"])

In [ ]:
#Drop the "Date" and "Heures" columns to keep one unique source of time data
data_2022 = data_2022.drop(columns=["Date", "Heures"])

**4. Temporal continuity verification**

- The temporal continuity of the dataset was verified by computing the time difference between consecutive timestamps.

- **The raw dataset from RTE has a 15-minute time step**, with 96 measurements per day. However, the 'Consommation' column is only measured every 30 minutes and since all rows with missing consumption data were dropped, **the remaining dataset naturally has a 30-minute time step, with 48 measurements per day**. This is the actual temporal resolution of the cleaned dataset.

- The distribution of time intervals was analyzed to ensure consistency and to detect any potential gaps in the time series.

- Additional checks were performed to identify duplicated timestamps and confirm the integrity of the temporal index.

In [ ]:
#Check the temporal continuity by analyzing the time differences between consecutive timestamps
Elapsed_time = data_2022.index.to_series().diff()
Elapsed_time.value_counts()

**4 - Analysis #1**

The results show a constant interval of 30 minutes throughout the entire dataset, which is consistent with the expected temporal resolution of RTE data.
The number of intervals (17519) is consistent with the total number of observations (17520), confirming the absence of temporal gaps.

In [ ]:
#Check the data for any possible duplicates
data_2022.duplicated().value_counts()

In [ ]:
#Last data check before proceeding with the variables creation
data_2022.head()

**5. Creation of useful variables**

Key system variables were constructed to represent the state of the electrical system.

At each timestamp (t), the system state is defined by two variables:

*   The total power **consumption**: the total power demand on the grid at instant t (in MW)

*   The total power **production**: the total power generation at all sources at instant t (in MW)

A useful insight would be detecting any **imbalance** through the difference Δ(t) between the production and the consumption:

*   Δ(t) > 0 : excess production --> more power is generated than consumed
*   Δ(t) < 0 : power deficit --> consumption exceeds production, blackout risk



In [ ]:
#Create the variables "Consumption"
Consumption = data_2022["Consommation"]
print(Consumption)

In [ ]:
#Create the variables "Production": Sum all generation sources to get total production at each timestamp
#Note: Pompage is excluded because it represents energy consumed to pump water back into reservoirs, not actual generation

Production = (data_2022["Fioul"]+data_2022["Charbon"]+data_2022["Gaz"]
+data_2022["Nucléaire"]+data_2022["Eolien"]+data_2022["Solaire"]
+data_2022["Hydraulique"]+data_2022["Bioénergies"])

print(Production)

In [ ]:
#Create the variable "Delta" (Positive = excess generation, Negative = power deficit)
Delta = Production - Consumption

print(Delta)

**5 - Analysis #1**

- From the results above, it seems that production consistently exceeds consumption across all timestamps.
- This is expected because physical exchanges and therefore exports (Ech. physiques) are not yet included in the balance.
- The Delta graph below will provide a more detailed view of these imbalances.

**6. Visualization**

In this section, visualizations were used to analyze the temporal behavior of the system.

- Time series plots were created to compare total production and consumption over the year.

- The power imbalance Delta(t) was also visualized to identify periods of stress or deviation from equilibrium.

In [ ]:
#Plot consumption vs production over time
fig, ax = plt.subplots()

Consumption.plot(ax=ax, label = "Consumption", title = "2022 France Power Data", xlabel = "Months")
Production.plot(ax=ax, label = "Production")

ax.legend(loc = "upper right")


**6 - Analysis #1**

We can observe on the graph above that:
- The system continuously adjusts production to match demand in real time.
- Winter months (Jan, Feb, Dec) show the highest peaks, driven by heating demand.
- Summer months (Jun and Aug) show the lowest consumption, with higher temperatures and reduced industrial activity.

In [ ]:
#Plot power imbalance Delta(t)
fig, ax = plt.subplots()
Delta.plot(ax=ax, title = "Delta 2022", xlabel ="Months")
plt.axhline(y=0, color= "black")

**6 - Analysis #2**

We can observe on the graph above that:

- Delta fluctuates around 0, meaning the system alternates between surplus and deficit throughout the year
- There are significant short-term imbalances (up to 15 000 MW),requiring constant system regulation through exports, imports, and generation adjustments.


In [ ]:
#Convert the index to a normal column to be visible in an csv file and for SQL for the next project sections
data_2022 = data_2022.reset_index()

#Save the cleaned dataset in the DATA file to be used in next project sections
data_2022.to_csv("/content/drive/MyDrive/Capstone Project - Data & Energy/Project Energy System/Data/clean_data_2022.csv", index=False)